# cudf-bench: stress grid (Step 1)

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

This notebook runs the full stress grid — pandas, Polars, and cuDF across sizes (1M/10M/30M rows) and key skew (uniform vs Zipf 1.5) — then **automatically downloads `results.csv` to your machine** at the end. Total ~15–25 min; don't close the tab while it runs (Colab wipes disconnected runtimes).

In [ ]:
# GPU check: a Tesla T4 should be listed. If not: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi

In [ ]:
try:
    import cudf
except ImportError:
    %pip install -q cudf-cu12
    import cudf
print("cudf", cudf.__version__)

In [ ]:
![ -d cudf-bench ] || git clone -q https://github.com/alexislowys/cudf-bench.git
%cd cudf-bench
%pip install -q -e .

## The grid

Same deterministic data for every backend (seed=0). pandas stops at 10M rows (it is the slow one — its cells take a few minutes, that is expected). Polars and cuDF also run 30M rows, pushing toward the T4's 16 GB.

`skew=1.5` means a handful of join/groupby keys dominate the table — the data shape that stresses hash-based algorithms.

In [ ]:
!python -m benchmark.runner --backend pandas --rows 1e6,1e7 --skew 0,1.5 --out results/results.csv

In [ ]:
!python -m benchmark.runner --backend polars --rows 1e6,1e7,3e7 --skew 0,1.5 --out results/results.csv

In [ ]:
!python -m benchmark.runner --backend cudf --rows 1e6,1e7,3e7 --skew 0,1.5 --out results/results.csv

## Quick look

Full analysis happens off-Colab; this is just a sanity glance. `speedup_polars_vs_pandas` and `speedup_cudf_vs_pandas` columns: higher = faster than pandas.

In [ ]:
import pandas as pd
pd.set_option("display.width", 160)
from benchmark.report import load_results, speedup_table

results = load_results("results/results.csv")
speedup_table(results).round(4)

## Auto-download results

Your browser downloads `results.csv` when this cell runs (allow the download if prompted). If it fails, use the Files panel: folder icon in the left edge strip → `cudf-bench/results/results.csv` → ⋮ → Download.

In [ ]:
from google.colab import files
files.download("/content/cudf-bench/results/results.csv")